In [2]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
import os
import asyncio
from openai.types.responses import ResponseTextDeltaEvent


In [3]:
load_dotenv(override=True)

True

In [4]:
instructions1 = "You are a professional sales agent for ComplAI that sells SOC2 compliance SaaS tool for compliance and audits, powered by AI. You write serious professional emails"
instructions2 = "You are a humorous engaging sales agent for ComplAI that sells SOC2 compliance SaaS tool for compliance and audits, powered by AI. You write fun engaging emails"
instructions3 = "You are a busy concise sales agent ComplAI that sells SOC2 compliance SaaS tool for compliance and audits, powered by AI. You write short punchy emails"

In [5]:
sales_agent1 = Agent(name="Professional_Sales_Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Engaging_Sales_Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Busy_Sales_Agent", instructions=instructions3, model="gpt-4o-mini")

In [6]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out to introduce you to ComplAI, a cutting-edge SaaS tool designed to streamline SOC 2 compliance and audit processes.

In today’s fast-paced business environment, ensuring compliance can often feel overwhelming and resource-intensive. ComplAI leverages advanced AI technology to automate and simplify the compliance journey, allowing your team to focus on what matters most—growing your business.

Here are a few key benefits of our solution:

- **Automated Compliance Workflows**: Reduce manual effort and eliminate human error with our intuitive platform.
- **Real-Time Monitoring**: Gain immediate insights and stay ahead of compliance requirements.
- **Seamless Integrations**: Easily connect with your existing tools and workflows for a cohesive experience.

I would love the opportunity to discuss how ComplAI can specifically address you

In [7]:
message = "Write a cold sales email"

with trace("Parallel sales agents writing emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
outputs = [result.final_output for result in results]
for output in outputs:
    print(output + "\n---\n")

Subject: Enhance Your SOC2 Compliance Efforts with AI-Powered Solutions

Dear [Recipient's Name],

I hope this message finds you well. I’m reaching out to introduce you to ComplAI, a leading provider of AI-driven tools designed to simplify and enhance your SOC2 compliance and audit processes.

In today's regulatory environment, maintaining compliance can be both time-consuming and complex. ComplAI streamlines this journey by leveraging advanced artificial intelligence to automate your compliance workflows, reduce manual effort, and mitigate risks effectively. Our platform not only helps you prepare for audits with ease but also provides continuous monitoring to ensure your systems remain compliant.

Here are a few key benefits of using ComplAI:

1. **Automated Documentation**: Significantly reduce the time spent on manual documentation and reporting.
2. **Real-Time Monitoring**: Stay ahead of compliance requirements with continuous insights and alerts.
3. **Comprehensive Support**: Acc

In [8]:
sales_picker = Agent(name= "sales_picker", instructions="You pick the best email from the given options. Imagine you are the customer and selecting the email that you would most likely reply to. Reply with the selected email.", model="gpt-4o-mini")

In [9]:
message = "Write a cold sales email"

with trace("Picking the best email"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(best.final_output)

I would select the second email:

---

Subject: Is Your SOC2 Compliance Giving You a Headache? 🤔💊

Hey [Recipient's First Name],

Ever feel like keeping up with SOC2 compliance is like trying to solve a Rubik's Cube blindfolded? You’re not alone! Many of us have been there, tangled in a maze of regulations and audits, all while wondering if we’ll ever make it out alive (or at least before that looming deadline!).

Introducing ComplAI – your new best friend in the wild world of compliance! 🎉✨

Imagine an AI-powered tool that not only simplifies your audits but also gives you superpowers to keep your compliance in check. Here’s what we bring to the table (and trust me, it’s more than just leftover office pizza):

- **Automated Audits**: Let our AI do the heavy lifting while you sit back and sip your coffee. ☕ 
- **Real-Time Monitoring**: Stay ahead of the compliance curve. No more midnight panic sessions!
- **User-Friendly Dashboards**: Because who doesn’t love a little bit of eye candy 

https://platform.openai.com/traces

In [10]:
@function_tool
def send_email(body: str):
    """Send out an email with the given body to all sales prospects"""
    # instead of SendGrid — just print
    print(f"\n{'='*50}")
    print("EMAIL SENT:")
    print(f"{'='*50}")
    print(body)
    print(f"{'='*50}\n")
    return {"status": "sent"}

In [11]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002819BC23880>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [12]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

sales_manager = Agent(name="Sales Manager", instructions="You pick the best email from 3 agents, then sends it using send_email tool", tools=tools, model="gpt-4o-mini")

In [13]:
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, "Send a cold sales email")
    print(result.final_output)


EMAIL SENT:
Subject: Introduction to Our SOC2 Compliance Solution

Dear [Recipient's Name],

I hope this message finds you well.

I am reaching out to introduce you to ComplAI, an advanced SOC2 compliance SaaS tool that leverages AI to simplify compliance and auditing processes. Our solution has successfully assisted numerous companies in not only achieving compliance but also enhancing operational efficiency and reducing costs.

I believe our tool could provide significant value to your organization as you navigate your compliance needs. Would you be open to a brief call to explore how ComplAI can specifically support your business goals?

Thank you for considering this opportunity. I look forward to the possibility of connecting.

Best regards,

[Your Name]  
[Your Position]  
ComplAI  
[Your Contact Information]  


EMAIL SENT:
Subject: Let’s Make Compliance Fun (Yes, You Read That Right!)

Hey [Recipient's Name]!

Hope you’re having a fantastic day! 🌟 I stumbled upon your company,

In [14]:
subject_instructions = "You write a subject line for a cold sales email. \
Write a subject that is likely to get a response."

html_instructions = "You convert a text email body to HTML. \
Convert the text email to clean, professional HTML."

subject_writer = Agent(
    name="Subject Writer",
    instructions=subject_instructions,
    model="gpt-4o-mini"
)

html_converter = Agent(
    name="HTML Converter", 
    instructions=html_instructions,
    model="gpt-4o-mini"
)

In [15]:
@function_tool
def send_html_email(subject: str, html_body: str):
    """Send out an email with the given subject and HTML body"""
    print(f"\n{'='*50}")
    print(f"EMAIL SENT")
    print(f"Subject: {subject}")
    print(f"{'='*50}")
    print(html_body)
    print(f"{'='*50}\n")
    return {"status": "sent"}

In [16]:
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject line for a cold sales email"
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a text email body to HTML"
)

emailer_tools = [subject_tool, html_tool, send_html_email]

emailer_instructions = "You are an email formatter and sender. \
You receive the body of an email to be sent. \
First use the subject_writer tool to write a subject. \
Then use the html_converter tool to convert the body to HTML. \
Finally use the send_html_email tool to send the email."

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=emailer_tools,
    handoff_description="Convert email to HTML and send",
    model="gpt-4o-mini"
)

sales_manager = Agent(
    name="Sales Manager",
    instructions="Pick the best cold sales email from the 3 sales agents, \
then hand off to the Email Manager to format and send it.",
    tools=[tool1, tool2, tool3],
    handoffs=[emailer_agent],
    model="gpt-4o-mini"
)

In [17]:
with trace("Sales Manager with Handoff"):
    result = await Runner.run(sales_manager, "Send a cold sales email")
    print(result.final_output)


EMAIL SENT
Subject: Exploring SOC2 Compliance Solutions for Your Business
<html>
<body>
<p>Hi [Recipient's Name],</p>
<p>I hope this message finds you well. I'm reaching out to introduce you to ComplAI, where we specialize in providing a comprehensive SOC2 compliance SaaS tool powered by AI. Our solution streamlines the compliance and audit process, helping businesses like yours efficiently manage and maintain their security standards.</p>
<p>Our clients, including [Client Example], have experienced significant reductions in compliance costs and improved audit readiness. I believe we can help you achieve similar results.</p>
<p>Would you be open to a brief call next week to discuss how our solution can support your compliance efforts?</p>
<p>Best regards,<br>
[Your Name]<br>
[Your Position]<br>
ComplAI<br>
[Your Phone Number]<br>
[Your Email]</p>
</body>
</html>

The cold sales email has been successfully sent! If you need anything else, just let me know.
